# PDF のページを画像のまま検索して VLM に答えさせる（マルチモーダル RAG）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/30_vision_language/multimodal_rag.ipynb)

NLP クックブック 30-07 ／ 画像と言語

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

表やグラフの多い PDF について RAG をしたい。テキストを抽出して検索する方法と、ページの画像をそのまま埋め込んで検索する方法を比べ、検索したページの画像を視覚言語モデル (VLM) に読ませて出典付きで答えさせたい。

In [1]:
# 所要時間の目安: T4 の Colab で 10〜15 分（大半はモデルのダウンロード。ColQwen2 と Qwen3-VL で計約 13.5GB）
!pip install -q pymupdf rank_bm25 fugashi unidic-lite

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import re, urllib.request
import numpy as np
import pandas as pd
import pymupdf
from PIL import Image

# 総務省統計局「家計調査報告（家計収支編）2025年（令和７年）平均結果の概要」（24 ページ、表とグラフが多い）
# 公共データ利用規約（PDL1.0、CC BY 4.0 互換）。出典: 総務省統計局ホームページ https://www.stat.go.jp/data/kakei/sokuhou/tsuki/index.html
urllib.request.urlretrieve("https://www.stat.go.jp/data/kakei/sokuhou/tsuki/pdf/fies_gaikyo2025.pdf", "kakei2025.pdf")
doc = pymupdf.open("kakei2025.pdf")
texts = [page.get_text() for page in doc]                                   # (A) 用: ページごとのテキスト（データの収集と入出力の章の「PDF からテキストを抽出する (PyMuPDF)」）
pages = []                                                                  # (B) 用: ページごとの画像
for page in doc:
    pix = page.get_pixmap(dpi=200)
    pages.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
print(len(doc), "ページ", pages[0].size, "| 文字数:", [len(t) for t in texts])
print("9 ページ目 (棒グラフ) から抽出したテキスト:", " ".join(texts[8].split())[:160])

24 ページ (1654, 2339) | 文字数: [124, 625, 611, 1453, 1127, 1709, 1392, 2302, 410, 1016, 1661, 1150, 1035, 1533, 1337, 1128, 771, 1927, 2520, 1141, 963, 1194, 1754, 677]
9 ページ目 (棒グラフ) から抽出したテキスト: 6 0.04 0.19 0.09 0.93 0.27 0.38 0.0 0.2 0.4 0.6 0.8 1.0 1.2 （％） 食 料 住 居 光 熱 ・ 水 道 家具・ 家事用 品 被 服 及 び 履 物 保 健 医 療 交 通 ・ 通 信 教 育 教 養 娯 楽 その他の消費支出 -0.36 -0.02 -0.06


In [3]:
# 評価セット: 質問と、答えが載っているページ番号（PDF の 1 始まりのページ番号。印刷されたページ番号 + 3）
# 「本文」は文章に答えが書いてある質問、「図表」は表やグラフを見ないと答えられない質問
eval_set = [
    ("本文", "2025年のエンゲル係数は何％でしたか。", [7]),
    ("本文", "3月に消費支出が増加したのはなぜですか。", [5]),
    ("本文", "大阪・関西万博の来場者数は約何人でしたか。", [6]),
    ("本文", "勤労者世帯の平均消費性向は何％で、前年から何ポイント変わりましたか。", [14]),
    ("本文", "単身世帯の消費支出は1か月平均いくらでしたか。", [17, 18]),
    ("本文", "65歳以上の夫婦のみの無職世帯の平均消費性向は何％でしたか。", [20, 22]),
    ("本文", "勤労者世帯の可処分所得は前年に比べて実質何％変化しましたか。", [12]),
    ("本文", "2020年に消費支出が大きく減少したのはなぜですか。", [4]),
    ("図表", "消費支出の実質増減率への寄与度が最も大きかった費目はどれで、寄与度はいくつでしたか。", [9]),
    ("図表", "電気代の実質増減率は何％でしたか。", [8]),
    ("図表", "勤労者世帯で消費支出額が最も多かった世帯主の年齢階級と、その金額は？", [13]),
    ("図表", "世帯主が60歳以上の勤労者世帯の割合は、2006年に何％でしたか。", [16]),
    ("図表", "勤労者世帯の黒字額は2018年にいくらでしたか。", [15]),
    ("図表", "勤労者世帯の消費支出に占める教養娯楽の割合は何％でしたか。", [15]),
    ("図表", "65歳以上の単身無職世帯の消費支出に占める食料の割合は何％でしたか。", [21, 22]),
    ("図表", "2025年に消費支出の名目増減率が最も高かったのは何月でしたか。", [5]),
    ("図表", "二人以上の世帯に占める無職世帯の割合は2025年に何％でしたか。", [16]),
]
questions = [q for _, q, _ in eval_set]

def gold_rank(ranking, gold):
    """ranking: ページ番号（0 始まり）を良い順に並べたもの。正解ページのうち最上位の順位を返す"""
    return min(list(ranking).index(g - 1) + 1 for g in gold)

def evaluate(rankings):
    rank = np.array([gold_rank(r, g) for r, (_, _, g) in zip(rankings, eval_set)])
    return {"Recall@1": (rank <= 1).mean(), "Recall@3": (rank <= 3).mean(), "Recall@5": (rank <= 5).mean(), "MRR": (1 / rank).mean()}

In [4]:
# (A) テキストの検索: BM25 と、E5 の埋め込み（ページを 400 字のチャンクに分け、最もよく合うチャンクの類似度をページの得点にする）
from fugashi import Tagger
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

tagger = Tagger()
tok = lambda s: [w.surface.lower() for w in tagger(s) if w.feature.pos1 not in ("助詞", "助動詞", "補助記号", "空白")]
bm25 = BM25Okapi([tok(t) for t in texts])
rank_bm25 = [np.argsort(-bm25.get_scores(tok(q))) for q in questions]

chunks, owner = [], []
for i, t in enumerate(texts):
    t = " ".join(t.split())
    for s in range(0, len(t), 300):            # 100 字ずつ重ねる
        chunks.append(t[s:s + 400]); owner.append(i)
owner = np.array(owner)
e5 = SentenceTransformer("intfloat/multilingual-e5-small")
sims = e5.encode(["query: " + q for q in questions], normalize_embeddings=True) @ \
       e5.encode(["passage: " + c for c in chunks], normalize_embeddings=True).T
page_sims = np.stack([sims[:, owner == i].max(axis=1) for i in range(len(texts))], axis=1)
rank_e5 = [np.argsort(-s) for s in page_sims]
pd.DataFrame({"BM25": evaluate(rank_bm25), "E5 (チャンク)": evaluate(rank_e5)}).round(3).T

,Recall@1,Recall@3,Recall@5,MRR
BM25,0.706,1.000,1.000,0.824
E5 (チャンク),0.647,0.824,0.824,0.751


In [5]:
# (B) ページ画像の検索: ColQwen2（Qwen2-VL-2B に ColBERT 式の late interaction を組み合わせたもの）
import torch
from transformers import ColQwen2ForRetrieval, ColQwen2Processor

BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 では float16
dtype = torch.bfloat16 if BF16 else torch.float16
torch.cuda.reset_peak_memory_stats()
name = "vidore/colqwen2-v1.0-hf"
retriever = ColQwen2ForRetrieval.from_pretrained(name, dtype=dtype, device_map="cuda").eval()   # 重みは約 4.4GB (Apache-2.0)
processor = ColQwen2Processor.from_pretrained(name)

def embed(batch):
    """ページ画像 (PIL) か質問 (str) のリストを、それぞれ（トークン数, 128）の行列のリストにする"""
    x = processor(images=batch) if isinstance(batch[0], Image.Image) else processor(text=batch)
    with torch.no_grad():
        out = retriever(**x.to(retriever.device)).embeddings
    return [e[m.bool()] for e, m in zip(out, x["attention_mask"])]     # パディングを除く

def maxsim(q, d):
    """質問の各トークンについて、ページの全トークンとの内積の最大値をとり、それを足し合わせる"""
    return (q @ d.T).max(dim=1).values.sum().item()

query_emb = embed(questions)
rankings = {}
for max_tokens in [768, 1536]:                       # 1 ページあたりの画像トークン数の上限（既定は 768）
    processor.image_processor.size = {"longest_edge": max_tokens * 28 * 28, "shortest_edge": 56 * 56}
    page_emb = [e for i in range(0, len(pages), 2) for e in embed(pages[i:i + 2])]
    scores = np.array([[maxsim(q, d) for d in page_emb] for q in query_emb])
    rankings[max_tokens] = [np.argsort(-s) for s in scores]
    n_vec = sum(len(e) for e in page_emb)
    print(f"上限 {max_tokens}: 索引 {len(page_emb)} ページ × 平均 {n_vec / len(page_emb):.0f} ベクトル × 128 次元 = {n_vec * 128 * 2 / 1e6:.1f} MB (16 ビット)")
print(f"(参考) E5 の索引: {len(chunks)} チャンク × 384 次元 = {len(chunks) * 384 * 2 / 1e6:.2f} MB")
print(f"VRAM の最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
del retriever; torch.cuda.empty_cache()              # VLM を載せる前に検索モデルを解放する（質問はすべて検索済み）

def rrf(rankings, k=60):                              # RAG の章の「検索の評価セットを作って手法を比べる」と同じ順位の融合
    scores = np.zeros(rankings[0].shape)
    for r in rankings:
        scores[np.arange(len(r))[:, None], r] += 1 / (k + np.arange(1, r.shape[1] + 1))
    return np.argsort(-scores, axis=1)

rank_img = rankings[1536]
pd.DataFrame({"BM25": evaluate(rank_bm25), "E5 (チャンク)": evaluate(rank_e5),
              "ColQwen2 (768 トークン)": evaluate(rankings[768]), "ColQwen2 (1536 トークン)": evaluate(rank_img),
              "BM25 + ColQwen2 (RRF)": evaluate(rrf([np.array(rank_bm25), np.array(rank_img)]))}).round(3).T

上限 768: 索引 24 ページ × 平均 747 ベクトル × 128 次元 = 4.6 MB (16 ビット)
上限 1536: 索引 24 ページ × 平均 1483 ベクトル × 128 次元 = 9.1 MB (16 ビット)
(参考) E5 の索引: 105 チャンク × 384 次元 = 0.08 MB
VRAM の最大使用量: 5.6 GB


,Recall@1,Recall@3,Recall@5,MRR
BM25,0.706,1.000,1.000,0.824
E5 (チャンク),0.647,0.824,0.824,0.751
ColQwen2 (768 トークン),0.588,0.824,0.882,0.693
ColQwen2 (1536 トークン),0.765,0.882,0.941,0.831
BM25 + ColQwen2 (RRF),0.706,0.882,0.941,0.817


In [6]:
# 質問ごとの正解ページの順位（1 が最良）。本文の質問と図表の質問で分けて見る
per_q = pd.DataFrame({"種類": [k for k, _, _ in eval_set], "質問": [q[:24] for q in questions],
                      **{n: [gold_rank(r, g) for r, (_, _, g) in zip(rs, eval_set)]
                         for n, rs in [("BM25", rank_bm25), ("E5", rank_e5), ("ColQwen2", rank_img)]}})
print(per_q.groupby("種類")[["BM25", "E5", "ColQwen2"]].agg(lambda r: (r == 1).mean()).round(2).rename_axis("Recall@1"))
per_q

          BM25    E5  ColQwen2
Recall@1                      
図表        0.56  0.44      0.67
本文        0.88  0.88      0.88


,種類,質問,BM25,E5,ColQwen2
0,本文,2025年のエンゲル係数は何％でしたか。,1,9,1
1,本文,3月に消費支出が増加したのはなぜですか。,1,1,1
2,本文,大阪・関西万博の来場者数は約何人でしたか。,1,1,1
3,本文,勤労者世帯の平均消費性向は何％で、前年から何ポイ,1,1,1
4,本文,単身世帯の消費支出は1か月平均いくらでしたか。,1,1,1
5,本文,65歳以上の夫婦のみの無職世帯の平均消費性向は何,1,1,1
6,本文,勤労者世帯の可処分所得は前年に比べて実質何％変化,1,1,1
7,本文,2020年に消費支出が大きく減少したのはなぜです,2,1,2
8,図表,消費支出の実質増減率への寄与度が最も大きかった費,1,1,1
9,図表,電気代の実質増減率は何％でしたか。,3,20,11


In [7]:
# 生成: 検索した上位 2 ページと質問を Qwen3-VL に渡し、資料の番号で出典を付けて答えさせる
# (B) はページの画像を、(A) は同じ VLM に BM25 で検索したページの抽出テキストを渡す
from transformers import AutoModelForImageTextToText, AutoProcessor

torch.cuda.reset_peak_memory_stats()
vlm_name = "Qwen/Qwen3-VL-4B-Instruct"
vlm = AutoModelForImageTextToText.from_pretrained(vlm_name, dtype=dtype, device_map="cuda").eval()   # 重みは約 8.9GB (Apache-2.0)
vlm_proc = AutoProcessor.from_pretrained(vlm_name)
vlm_proc.image_processor.size = {"longest_edge": 2048 * 32 * 32, "shortest_edge": 256 * 256}   # 1 ページを最大 2048 トークンに抑える

INSTRUCTION = ("上の資料だけに基づいて、質問に日本語で簡潔に答えてください。根拠にした資料の番号を文末に [1] のように付けてください。"
               "資料に答えがなければ「わかりません」と答えてください。\n質問: ")

def answer(qi, use_image=True, k=2):
    hits = [int(p) for p in (rank_img if use_image else rank_bm25)[qi][:k]]
    content = []
    for n, p in enumerate(hits, 1):
        if use_image:
            content += [{"type": "text", "text": f"資料 [{n}]:"}, {"type": "image", "image": pages[p]}]
        else:
            content += [{"type": "text", "text": f"資料 [{n}]: {' '.join(texts[p].split())}\n"}]
    content.append({"type": "text", "text": INSTRUCTION + questions[qi]})
    inputs = vlm_proc.apply_chat_template([{"role": "user", "content": content}], tokenize=True, add_generation_prompt=True,
                                          return_dict=True, return_tensors="pt").to(vlm.device)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=128, do_sample=False)
    text = vlm_proc.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    cited = [hits[int(n) - 1] + 1 for n in re.findall(r"\[(\d+)\]", text) if 1 <= int(n) <= k]   # 資料の番号 → PDF のページ番号
    return " ".join(text.split()), [p + 1 for p in hits], cited

for qi in [0, 3, 8, 10, 12, 15]:
    kind, q, gold = eval_set[qi]
    print(f"[{kind}] Q: {q} (正解のページ: {gold})")
    for label, use_image in [("(A) テキスト", False), ("(B) 画像", True)]:
        text, hits, cited = answer(qi, use_image)
        mark = "出典○" if cited and set(cited) <= set(gold) else "出典×"     # 出典が正解のページかだけを見る（答えの正しさは見ていない）
        print(f"  {label}  検索: p.{hits}  出典: p.{cited} {mark} | {text[:150]}")
print(f"VRAM の最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

[本文] Q: 2025年のエンゲル係数は何％でしたか。 (正解のページ: [7])
  (A) テキスト  検索: p.[7, 4]  出典: p.[7, 7, 4] 出典× | 2025年のエンゲル係数は資料[1]に記載されていません。資料[1]ではエンゲル係数の値が「前年から0.3ポイントの上昇」と述べられていますが、その具体的な数値（例：28.6％）は2024年時点のものであり、2025年の値は記載されていません。資料[2]にも2025年のエンゲル係数の数値は含まれてい
  (B) 画像  検索: p.[7, 4]  出典: p.[7] 出典○ | 2025年のエンゲル係数は28.6％でした [1]。
[本文] Q: 勤労者世帯の平均消費性向は何％で、前年から何ポイント変わりましたか。 (正解のページ: [14])
  (A) テキスト  検索: p.[14, 3]  出典: p.[14] 出典○ | 勤労者世帯の平均消費性向は65.0％で、前年から2.8ポイント上昇しました [1]。
  (B) 画像  検索: p.[14, 3]  出典: p.[14] 出典○ | 勤労者世帯の平均消費性向は65.0％で、前年比2.8ポイント上昇しました [1]。
[図表] Q: 消費支出の実質増減率への寄与度が最も大きかった費目はどれで、寄与度はいくつでしたか。 (正解のページ: [9])
  (A) テキスト  検索: p.[9, 18]  出典: p.[18] 出典× | 消費支出の実質増減率への寄与度が最も大きかった費目は「教育」で、寄与度は3.7%でした。 [2]
  (B) 画像  検索: p.[9, 18]  出典: p.[18] 出典× | 交通・通信が最も大きかったで、寄与度は0.93%でした [2]。
[図表] Q: 勤労者世帯で消費支出額が最も多かった世帯主の年齢階級と、その金額は？ (正解のページ: [13])
  (A) テキスト  検索: p.[23, 20]  出典: p.[23, 20, 23, 23] 出典× | 資料に勤労者世帯の消費支出額の年齢階級別データは含まれていません。資料[1]と[2]の「参考４」では、65歳以上の無職世帯の消費支出が示されていますが、これは勤労者世帯とは異なります。また、資料[1]の図や表のタイトルからも、勤

- 画像のまま検索する方法（ColPali 系）は、ページの画像を VLM に通し、画像のパッチごとのベクトル（ここでは 1 ページ約 1500 個 × 128 次元）を索引にする。質問もトークンごとのベクトルにし、質問の各トークンについてページ内で最も近いベクトルとの内積をとって足し合わせる（MaxSim、late interaction）。文全体を 1 本のベクトルにする E5 より細かく照合できるが、索引はこの PDF でも 9.1MB と E5 (0.08MB) の 100 倍以上になる。1 ページの画像トークン数の上限を既定の 768 から 1536 にすると、Recall@1 は 0.588 から 0.765 に上がった（BM25 は 0.706）。日本語の小さな文字や表の数字を読むには解像度が要る（この章の「PDF のページを画像にして視覚言語モデルで読む（表とグラフ）」も参照）。注意書きの数値は、特に断らない限り T4 と同じ float16 で動かしたときのもの。
- この PDF はテキスト層があるので、テキストの検索も強い。BM25 は Recall@3 が 1.0 で、本文に答えがある質問の Recall@1 は 3 つの方法とも 0.88 で同じだった。差が出たのは図表の質問で、Recall@1 は BM25 が 0.56、E5 が 0.44、ColQwen2 が 0.67。表の項目名は「交 通 ・ 通 信」のように 1 字ずつ区切られて抽出され、グラフの数値は位置の情報なしに並ぶ。BM25 と ColQwen2 を RRF で融合しても ColQwen2 単独とほぼ同じで（MRR は float16 で 0.817、bfloat16 で 0.852。ColQwen2 単独は 0.831 と 0.841）、上回るとは言えない。float16 では融合後の Recall@1 が 0.706 に下がった。スキャンした PDF ならテキストは抽出できないので、画像の検索しか使えない。
- 生成でも差が出た。同じ Qwen3-VL に (A) 抽出したテキストを渡すと、6 問中はっきり正しく答えたのは平均消費性向の 1 問だけだった。エンゲル係数では答え (28.6%) のあるページを渡したのに「資料[1]に記載されていません」と答え、寄与度のグラフでは「教育、3.7%」と誤り、名目増減率では「12 月」と「5 月」を併記した。(B) ページの画像を渡すと、6 問中 4 問（エンゲル係数、平均消費性向、寄与度の「交通・通信 0.93」、名目増減率が最も高い「5 月 8.9%」）に正しく答えた。
- 画像を渡しても間違える。「2018 年の黒字額」は、正しいページを渡したのに、積み上げ棒グラフの同じ年の「その他」の値（-23.6 千円）を答えた（正しくは 139.8 千円。出典は正しいので「出典○」になる）。「消費支出額が最も多い年齢階級」は、検索で正解のページ（13 ページ）が 5 位になり、無職世帯のページ（20 ページ）からもっともらしい答えを作った。答えだけを見ても検索の失敗とは気づけないので、RAG の章の「RAG の失敗を検索と生成に切り分けて分析する」のように、検索と生成を分けて評価する。
- 出典は、ページの中に印刷されたページ番号（この PDF では PDF のページ番号と 3 ずれる）と紛れないように、資料の番号 ([1], [2]) で付けさせ、コードでページ番号に戻している。それでも、正しく答えたのに別の資料の番号を付けることがある（寄与度と名目増減率の質問）。RAG の章の「回答に出典を付けさせ、その出典を検証する」のように出典を検証する。
- 検索の単位はページ。1 ページに図が 2 つ以上ある（15、16 ページ）と、VLM は関係のない図も読むことになる。ページ画像は 1 枚で最大 2048 トークンを使うので、VLM に渡せるページは数枚まで。
- VRAM の最大使用量は ColQwen2 が 5.6GB、Qwen3-VL-4B が 10.6GB（2 ページ、1 ページ最大 2048 トークン）。ColQwen2 の重み (4.4GB) を残したまま VLM を動かすと約 15GB で、T4 で使える量にほぼ達する。そこで質問の検索を先にすべて済ませ、検索モデルを解放してから VLM を読み込んでいる。
- ColQwen2（vidore/colqwen2-v1.0-hf、Apache-2.0）は Qwen2-VL-2B-Instruct をもとにしたモデルで、モデルカードによると検索のための学習データは英語だけ（もとの Qwen2-VL の事前学習には多言語のデータが含まれる）。この `-hf` 版は transformers だけで動く。データは総務省統計局「家計調査報告（家計収支編）2025年（令和７年）平均結果の概要」（公共データ利用規約 第1.0版、CC BY 4.0 互換）。評価セットの 17 問と正解のページは筆者が PDF を読んで作ったもので、1 問で Recall が 0.06 動く。